# **Task 1: News Topic Classifier Using BERT**

**Objective:** Fine-tune BERT on the AG News dataset.

**Environment & Tools:** Google Colab, Python, Hugging Face Transformers, Datasets, PyTorch, Gradio.

## **Step 1: Install Required Libraries**
**Purpose:** Install the required packages.

In [3]:
!pip -q install -U transformers datasets evaluate accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.9 MB/s eta 0:00:00


## **Step 2: Import Libraries**
**Purpose:** Import required modules.

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

## **Step 3: Load Dataset**
**Purpose:** Load AG News. If the default loader fails, the fallback loads the same dataset from Parquet files.

In [6]:
try:
    dataset=load_dataset("ag_news")
except Exception:
    dataset=load_dataset("parquet",data_files={
      "train":"https://huggingface.co/datasets/fancyzhx/ag_news/resolve/main/data/train-00000-of-00001.parquet",
      "test":"https://huggingface.co/datasets/fancyzhx/ag_news/resolve/main/data/test-00000-of-00001.parquet"})
print(dataset)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


## **Step 4: Explore Dataset**
**Purpose:** View sample records and dataset information.

In [7]:
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## **Step 5: Tokenization**
**Purpose:** Convert text into tokens that BERT understands.

In [8]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
tokenized=dataset.map(tokenize, batched=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

## **Step 6: Load BERT Model**
**Purpose:** Load a pretrained BERT model for 4-class classification.

In [9]:
model=AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=4)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## **Step 7: Configure Training**
**Purpose:** Define training settings.

In [10]:
!pip install -q transformers datasets evaluate accelerate

In [11]:
import evaluate
print("Evaluate imported successfully!")

Evaluate imported successfully!


In [14]:
import transformers
print(transformers.__version__)

5.12.1


In [12]:
import numpy as np
import evaluate
from transformers import Trainer, TrainingArguments

# Load evaluation metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(
            predictions=preds,
            references=labels
        )["accuracy"],

        "f1": f1.compute(
            predictions=preds,
            references=labels,
            average="weighted"
        )["f1"]
    }

# Training arguments
args = TrainingArguments(
    output_dir="results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_strategy="epoch",
    load_best_model_at_end=False,
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,      # <-- replaces tokenizer=
    compute_metrics=compute_metrics
)

## **Step 8: Fine-tune BERT**
**Purpose:** Train the model.

In [13]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("Current Device:", torch.cuda.current_device())
print("GPU:", torch.cuda.get_device_name(0))
print("Model Device:", next(model.parameters()).device)

CUDA Available: True
Current Device: 0
GPU: Tesla T4
Model Device: cuda:0


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.222273,0.212966,0.945789,0.945832


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15000, training_loss=0.22227277018229166, metrics={'train_runtime': 3002.2931, 'train_samples_per_second': 39.969, 'train_steps_per_second': 4.996, 'total_flos': 1.4078799161376768e+16, 'train_loss': 0.22227277018229166, 'epoch': 1.0})

## **Step 9: Evaluate Model**
**Purpose:** Measure Accuracy and F1-score.

In [16]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.222273,0.212966,1,0.945789,0.945832


{'eval_loss': 0.21296614408493042,
 'eval_accuracy': 0.9457894736842105,
 'eval_f1': 0.9458324753752885}

## **Step 10: Save Model**
**Purpose:** Save the trained model and tokenizer.

In [17]:
trainer.save_model("bert_agnews_model")
tokenizer.save_pretrained("bert_agnews_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_agnews_model/tokenizer_config.json', 'bert_agnews_model/tokenizer.json')

## **Step 11: Test Custom Headline**
**Purpose:** Predict a category for new input.

In [19]:
import torch

# Put model in evaluation mode
model.eval()

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Custom headline
text = "Apple launches a new AI chip."

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# Move inputs to GPU
inputs = {k: v.to(device) for k, v in inputs.items()}

# Predict
with torch.no_grad():
    outputs = model(**inputs)
    pred = outputs.logits.argmax(dim=-1).item()

# Labels
labels = ["World", "Sports", "Business", "Sci/Tech"]

print("Headline:", text)
print("Predicted Topic:", labels[pred])

Headline: Apple launches a new AI chip.
Predicted Topic: Sci/Tech


## **Step 12: Gradio Demo**
**Purpose:** Create a simple web interface.

In [21]:
import gradio as gr
import torch

# Put model in evaluation mode
model.eval()

# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Make sure model is on GPU
model = model.to(device)

# Label names
labels = ["World", "Sports", "Business", "Sci/Tech"]

def predict(text):
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Prediction
    with torch.no_grad():
        outputs = model(**inputs)
        pred = outputs.logits.argmax(dim=-1).item()

    return labels[pred]

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Enter a news headline..."
    ),
    outputs=gr.Textbox(label="Predicted Category"),
    title="AG News Classifier"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9e7593852203f9f23f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## **Results**
- BERT was fine-tuned on the AG News dataset.
- Performance was evaluated using Accuracy and F1-score.
- A Gradio interface allows live predictions.

## **Conclusion**
The project demonstrates transformer-based text classification using BERT, transfer learning, evaluation metrics, and lightweight deployment.